# Выгрузка данных для аналитической витрины из основной

## Ипорты

In [1]:
import pandas as pd
import numpy as np

## Загружаем таблицы

In [2]:
clients = pd.read_excel('nerd_analytics_v19.xlsx', sheet_name='clients')
employees = pd.read_excel('nerd_analytics_v19.xlsx', sheet_name='employees')
tickets = pd.read_excel('nerd_analytics_v19.xlsx', sheet_name='tickets')
attachments = pd.read_excel('nerd_analytics_v19.xlsx', sheet_name='attachments')
reviews = pd.read_excel('nerd_analytics_v19.xlsx', sheet_name='reviews')
notifications = pd.read_excel('nerd_analytics_v19.xlsx', sheet_name='notifications')
chat_history = pd.read_excel('nerd_analytics_v19.xlsx', sheet_name='chat_history')

In [3]:
chat_history['row_num'] = chat_history.groupby('chat_id').cumcount()
chat_history['admin_met'] = (chat_history['role'] == 'admin').groupby(chat_history['chat_id']).cumsum()
chat_history['last_time'] = chat_history['created_at'].shift(1)
chat_history['last_time'] = chat_history.apply(lambda row: np.nan if row['row_num'] == 0 else row['last_time'], axis=1)
chat_history['time_to_ans'] = (pd.to_datetime(chat_history['created_at']) - pd.to_datetime(chat_history['last_time'])).dt.total_seconds()

## Дашборд 1

In [4]:
dash1 = tickets.rename(columns={
    'id': 'ticket_id',
    'date': 'created_at',
    'final_category': 'category',
})
dash1['closed_at'] = pd.to_datetime(dash1['closed_at'])
dash1['created_at'] = pd.to_datetime(dash1['created_at'])

In [5]:
correct_shape = dash1.shape[0]

In [6]:
# Джойн сотрудников
dash1 = pd.merge(dash1, employees[['id', 'full_name']].rename(columns={'full_name': 'admin'}), how='left', left_on='responsible_id', right_on='id').drop('id', axis=1)

In [7]:
# Джойн клиентов
dash1 = pd.merge(dash1, clients[['id', 'age', 'city']].rename(columns={'age': 'age_group'}), how='left', left_on='client_id', right_on='id').drop('id', axis=1)

In [8]:
# Джойн отзывов
dash1 = pd.merge(dash1, reviews[['ticket_id', 'rating']], how='left', on='ticket_id')

In [9]:
bins = [18, 25, 35, 45, 55, 100]
labels = ['18-25', '25-35', '35-45', '45-55', '55+']

dash1['age_group'] = pd.cut(dash1['age_group'], bins=bins, labels=labels, right=False)

In [10]:
dash1['ttl_hours'] = (dash1['closed_at'] - dash1['created_at']).dt.total_seconds() / 3600
dash1['is_reopened'] = dash1['reopened_count'] > 0
dash1['created_at'] = dash1['created_at'].dt.date
dash1['closed_at'] = dash1['closed_at'].dt.date

In [11]:
dash1 = dash1[['ticket_id', 'created_at', 'closed_at', 'product', 'category', 'status', 'priority',
                 'admin', 'city', 'age_group', 'ttl_hours', 'is_reopened', 'reopened_count', 'rating']]

In [12]:
assert dash1.shape[0] == correct_shape

## Дашборд 2

In [13]:
chat_history['created_at'] = pd.to_datetime(chat_history['created_at'])
chat_history = chat_history.sort_values(['chat_id', 'created_at'])

In [14]:
dash2 = chat_history[['chat_id', 'ticket_id', 'product', 'category', 'resolved_by_ai']].drop_duplicates()
dash2['escalated_to_human'] = ~dash2['resolved_by_ai']

In [15]:
correct_shape = dash2.shape[0]

In [16]:
dash2 = dash2.rename(columns={'chat_id': 'dialog_id', 'resolved_by_ai' : 'is_resolved_by_ai', 'category': 'ai_category_suggested'})

In [17]:
# Джойн тикетов
dash2 = pd.merge(dash2, tickets[['id', 'final_category', 'date', 'closed_at']].rename(columns={'id': 'ticket_id'}), how='left', on='ticket_id')
dash2['resolution_time_human_min'] = (pd.to_datetime(dash2['closed_at']) - pd.to_datetime(dash2['date'])).dt.total_seconds()/60
dash2 = dash2.drop(['date'], axis=1)

In [18]:
dash2.loc[(dash2['escalated_to_human'] == True) & (dash2['ai_category_suggested'] == dash2['final_category']), 'admin_changed_category'] = False
dash2.loc[(dash2['escalated_to_human'] == True) & (dash2['ai_category_suggested'] != dash2['final_category']), 'admin_changed_category'] = True

In [19]:
dash2['is_resolved_by_ai'] = dash2['ticket_id'].isna()
dash2['escalated_to_human'] = dash2['ticket_id'].notna()

In [20]:
result = chat_history[chat_history['admin_met'] == 0].copy()
result = result.drop(columns=['row_num', 'admin_met'])

In [21]:
# Сообщения до эскалации
pivot = pd.pivot_table(index='chat_id', values='message', aggfunc='count', data=result[result['role'] == 'ai']).reset_index().rename(columns={'chat_id': 'dialog_id', 'message': 'msg_count_before_escalation'})

In [22]:
# Среднее время ответа ИИ
pivot2 = result[result['role'] == 'ai'].groupby('chat_id')['time_to_ans'].mean().reset_index()
pivot2 = pivot2.rename(columns={'chat_id': 'dialog_id', 'time_to_ans': 'ai_response_sec'})

In [23]:
# Даты первого и последнего сообщений
dates = result.groupby('chat_id')['created_at'].agg(['min', 'max']).reset_index()
dates['resolution_time_ai_min'] = (dates['max'] - dates['min']).dt.total_seconds()/60

In [24]:
dash2 = pd.merge(dash2, dates[['chat_id', 'min','resolution_time_ai_min']].rename(columns={'chat_id': 'dialog_id', 'min': 'date'}), how='left', on='dialog_id')
dash2 = pd.merge(dash2, pivot, how='left', on='dialog_id')
dash2 = pd.merge(dash2, pivot2, how='left', on='dialog_id')

In [25]:
dash2 = dash2.rename(columns={'created_at': 'date'})
dash2['date'] = dash2['date'].dt.date
dash2 = dash2[['dialog_id', 'ticket_id', 'date', 'product', 'msg_count_before_escalation',
               'is_resolved_by_ai', 'escalated_to_human', 'ai_response_sec', 'ai_category_suggested',
               'admin_changed_category', 'final_category', 'resolution_time_ai_min', 'resolution_time_human_min']]

In [26]:
assert correct_shape == dash2.shape[0]

## Дашборд 3

In [27]:
dash3 = tickets[['id', 'date', 'responsible_id', 'product', 'final_category', 'priority', 'sla_ttfr_min', 'sla_ttr_min', 'closed_at']]

In [28]:
correct_shape = dash3.shape[0]

In [29]:
dash3 = dash3.rename(columns={'id': 'ticket_id', 'final_category': 'category'})

In [30]:
dash3['hour'] = dash3['date'].dt.hour
dash3['day_of_week'] = dash3['date'].dt.dayofweek

In [31]:
sl_week = {
    0: 'понедельник',
    1: 'вторник',
    2: 'среда',
    3: 'четверг',
    4: 'пятница',
    5: 'суббота',
    6: 'воскресенье'
}

In [32]:
dash3['day_of_week'] = dash3['day_of_week'].map(sl_week)

In [33]:
dash3['ttr_min'] = (pd.to_datetime(dash3['closed_at']) - pd.to_datetime(dash3['date'])).dt.total_seconds()/60

In [34]:
# Время первого ответа админа
ttr = chat_history[(chat_history['admin_met'] == 1) & (chat_history['role'] == 'admin')]
ttr = ttr[['ticket_id', 'time_to_ans']]
ttr = ttr.rename(columns={'time_to_ans': 'ttfr_min'})

In [35]:
dash3 = dash3.merge(ttr, how='left', on='ticket_id')
dash3['ttfr_min'] = dash3['ttfr_min'] / 60

In [36]:
dash3['ttfr_met'] = dash3['ttfr_min'] <= dash3['sla_ttfr_min']
dash3['ttr_met'] = dash3['ttr_min'] <= dash3['sla_ttr_min']
dash3['tickets_closed'] = dash3['closed_at'].isna()

In [37]:
# Джойним отзывы
dash3 = dash3.merge(reviews[['ticket_id', 'rating']], how='left', on='ticket_id')

In [38]:
# Джойним сотрудников
dash3 = dash3.merge(employees[['id', 'full_name']].rename(columns={'id': 'responsible_id'}), how='left', on='responsible_id')

In [39]:
dash3['date'] = dash3['date'].dt.date
dash3 = dash3.rename(columns={'full_name': 'admin', 'rating': 'rating_from_user'})
dash3 = dash3[['ticket_id', 'date', 'hour', 'day_of_week', 'product', 'category', 'priority', 'admin', 'ttfr_min', 'ttr_min', 'sla_ttfr_min', 'sla_ttr_min', 'ttfr_met', 'ttr_met', 'tickets_closed', 'rating_from_user']]

In [40]:
assert correct_shape == dash3.shape[0]

# Дашборд 4

In [41]:
dash4 = clients[['id', 'created_at', 'gender', 'age', 'city']]
dash4 = dash4.rename(columns={
    'id': 'user_id',
    'created_at': 'registration_date',
})
bins = [18, 25, 35, 45, 55, 100]
labels = ['18-25', '25-35', '35-45', '45-55', '55+']

dash4['age_group'] = pd.cut(dash4['age'], bins=bins, labels=labels, right=False)

In [42]:
correct_shape = dash4.shape[0]

In [43]:
# Дата первого и последнего тикета
pivot = tickets.groupby('client_id')['date'].agg(['min','max']).reset_index().rename(columns={
    'min': 'first_ticket_date',
    'max': 'last_ticket_date',
    'client_id': 'user_id',
})
# Всего тикетов
pivot2 = tickets.groupby('client_id').agg({'id': 'count'}).reset_index().rename(columns={'client_id': 'user_id', 'id': 'total_tickets'})
# Всего открытых тикетов
pivot3= tickets[tickets['closed_at'].isna()].groupby('client_id').agg({'id': 'count'}).reset_index().rename(columns={'client_id': 'user_id', 'id': 'open_tickets'})
# Всего закрытых тикетов
pivot4= tickets[tickets['closed_at'].notna()].groupby('client_id').agg({'id': 'count'}).reset_index().rename(columns={'client_id': 'user_id', 'id': 'closed_tickets'})
# Список продуктов клиента (по тикетам)
pivot5= result = tickets[['client_id', 'product']].drop_duplicates().groupby('client_id')['product'].apply(list).reset_index().rename(columns={'client_id': 'user_id'})

In [44]:
dash4 = dash4.merge(pivot, how='left', on='user_id')
dash4 = dash4.merge(pivot2, how='left', on='user_id')
dash4 = dash4.merge(pivot3, how='left', on='user_id')
dash4 = dash4.merge(pivot4, how='left', on='user_id')
dash4 = dash4.merge(pivot5, how='left', on='user_id')

In [45]:
# Последний тикет это возвращение в течении 7, 14 или 30 дней?
tickets_sorted = tickets.sort_values(['client_id', 'date'])
tickets_sorted['days_since_prev'] = tickets_sorted.groupby('client_id')['date'].diff().dt.days
result = tickets_sorted.groupby('client_id').last().reset_index()
result = result[['client_id', 'date', 'days_since_prev']]
result['retention_7d'] = result['days_since_prev'] <= 7
result['retention_14d'] = (result['days_since_prev'] <= 14) & (result['retention_7d'] == False)
result['retention_30d'] = (result['days_since_prev'] <= 30) & (result['retention_7d'] == False) & (result['retention_14d'] == False)
result = result.rename(columns={'client_id': 'user_id'})

In [46]:
dash4 = dash4.merge(result, how='left', on='user_id')

In [47]:
dash4['date'] = dash4['date'].dt.date
dash4 = dash4[['user_id', 'registration_date', 'gender', 'age_group', 'city', 'product', 'total_tickets',
               'open_tickets', 'closed_tickets', 'first_ticket_date', 'last_ticket_date',
               'retention_7d', 'retention_14d', 'retention_30d']]

In [48]:
assert correct_shape == dash4.shape[0]

## Дашборд 5

In [49]:
dash5 = reviews[['id', 'client_id', 'ticket_id', 'created_at', 'product', 'final_category', 'rating', 'comment', 'sentiment', 'keywords_positive', 'keywords_negative']]

In [50]:
# Джойним клиентов
dash5 = dash5.merge(clients[['id', 'city', 'age']].rename(columns={'id': 'client_id'}), how='left', on='client_id')

In [51]:
bins = [18, 25, 35, 45, 55, 100]
labels = ['18-25', '25-35', '35-45', '45-55', '55+']

dash5['age_group'] = pd.cut(dash5['age'], bins=bins, labels=labels, right=False)

In [52]:
dash5 = dash5.rename(columns={
    'id': 'review_id',
    'created_at': 'date',
    'final_category': 'category',
    'comment': 'review_text',
})

In [53]:
dash5['date'] = pd.to_datetime(dash5['date']).dt.date

In [54]:
dash5 = dash5[['review_id', 'ticket_id', 'date', 'product', 'category', 'city', 'age_group',
               'rating', 'review_text', 'sentiment', 'keywords_positive', 'keywords_negative']]

## Дашборд 6

In [55]:
dash6 = tickets[['id', 'date', 'product', 'final_category', 'priority', 'keywords', 'client_id', 'closed_at']]
dash6 = dash6.rename(columns={'id': 'ticket_id', 'final_category': 'category'})

In [56]:
correct_shape = dash6.shape[0]

In [57]:
dash6['hour'] = dash6['date'].dt.hour
dash6['day_of_week'] = dash6['date'].dt.dayofweek
sl_week = {
    0: 'понедельник',
    1: 'вторник',
    2: 'среда',
    3: 'четверг',
    4: 'пятница',
    5: 'суббота',
    6: 'воскресенье'
}
dash6['day_of_week'] = dash6['day_of_week'].map(sl_week)

In [58]:
# Джойним клиентов
dash6 = pd.merge(dash6, clients[['id', 'age', 'city']], left_on='client_id', right_on='id', how='left')

In [59]:
bins = [18, 25, 35, 45, 55, 100]
labels = ['18-25', '25-35', '35-45', '45-55', '55+']

dash6['age_group'] = pd.cut(dash6['age'], bins=bins, labels=labels, right=False)
dash6['ttl_hours'] = (pd.to_datetime(dash6['closed_at']) - pd.to_datetime(dash6['date'])).dt.total_seconds() / 3600

In [60]:
# Скользящее среднее число обращений по продукту + категории

daily_counts = dash6.groupby(['date', 'product', 'category']).size().reset_index(name='cnt')
daily_counts = daily_counts.sort_values(['product', 'category', 'date'])

def calc_moving_stats(group, window=14):
    group = group.copy()
    group['moving_avg'] = group['cnt'].rolling(window=window, min_periods=1).mean()
    group['moving_std'] = group['cnt'].rolling(window=window, min_periods=1).std()
    return group

daily_stats = daily_counts.groupby(['product', 'category']).apply(calc_moving_stats).reset_index()
daily_stats['moving_std'] = daily_stats['moving_std'].fillna(0)
daily_stats['is_anomaly'] = (
    (daily_stats['cnt'] > daily_stats['moving_avg'] + 2 * daily_stats['moving_std']) |
    (daily_stats['cnt'] < daily_stats['moving_avg'] - 2 * daily_stats['moving_std'])
).astype(int)

dash6 = dash6.merge(
    daily_stats[['date', 'product', 'category', 'is_anomaly']],
    on=['date', 'product', 'category'],
    how='left'
)

In [61]:
dash6['date'] = dash6['date'].dt.date
dash6 = dash6[['ticket_id', 'date', 'hour', 'day_of_week', 'product', 'category', 'priority',
               'city', 'age_group', 'keywords', 'ttl_hours', 'is_anomaly']]

In [62]:
assert correct_shape == dash6.shape[0]